<a href="https://colab.research.google.com/github/corcondor/geocpu-diagnostics/blob/main/research/v48_balanced_augmentation_20261108/GeoCPU_v48_Colab_40500_to_balanced.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
**診断改善（11月8日）**: 失敗時は子プロセスの本当のTracebackを末尾80行とともに表示し、Driveの `_v48_t4_diagnostics/` にログを保存します。元の学習量とモデルは変更しません。


# GeoCPU v48｜T4ハイメモリ・学習量を削らない継続学習

Colabで**T4 GPU・ハイメモリ**を選び、次のセルを実行してください。

**学習量を削減しません。** 元の422,500行＋v47追加848行は保持し、新規4,096行を加えます。基準となる40,500ステップのモデル・Adam・RNGはDriveから検証し、別の出力先へ保存します。

学習の論理予算は**36,864トークン／更新**、更新数は10,000～20,000を事前検査で決定、バッチ上限は64のままです。GPUメモリ節約は、16,384トークンを初期単位とする計算分割・活性再計算・FP16動的スケーリングで行います。分割がOOMになる場合は同一バッチをより小さな単位で再計算し、**問題を捨てず、更新回数を水増ししません**。

**T4 High-RAMのRAM増量はCPU側です。GPU VRAMは16GBのまま**で、A100と数値計算精度が同一になるわけではありません。

GitHubはPublicのためトークン不要。最初に学習データ再構成・グループ別の旧教師トークン非減少・4カテゴリの均衡をCPU事前検査します。いずれかに失敗すれば**GPU学習は開始しません**。

このT4版はGitHub Actionsの静的・CPU検証対象ですが、実Colab T4での完走とモデル評価は未検証です。


In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess
import sys
import tempfile
import torch

drive.mount("/content/drive")

assert torch.cuda.is_available(), "ColabでT4 GPUを選んでください。"
GPU_NAME = torch.cuda.get_device_name(0)
VRAM = torch.cuda.get_device_properties(0).total_memory / (1024**3)
assert "T4" in GPU_NAME, f"T4専用コードです。現在のGPU: {GPU_NAME}"
assert VRAM >= 14, f"GPU VRAM不足: {VRAM:.1f}GiB"

print("GPU:", GPU_NAME)
print(f"GPU VRAM: {VRAM:.1f}GiB （ハイメモリ設定でも16GiB前後）")
print("PyTorch:", torch.__version__)

REPO = Path(tempfile.mkdtemp(prefix="geocpu_v48_t4_repo_", dir="/content")) / "geocpu-diagnostics"
subprocess.run([
    "git", "clone", "--depth", "1", "--branch", "main",
    "https://github.com/corcondor/geocpu-diagnostics.git",
    str(REPO)
], check=True)

STUDY = Path("/content/drive/MyDrive/GeoCPU_v47_fullmodel_20261106")
ARCHIVE = STUDY / "GeoCPU_practical_training_logging_20261107" / "GeoCPU_practical_training_logging_20261107_10k.zip"
OUT = STUDY / "v48_T4_highram_balanced_from40500_20261108"
RUNNER = REPO / "research/v48_balanced_augmentation_20261108/train_from_40500.py"

assert ARCHIVE.is_file(), f"元v47 ZIPが見つかりません: {ARCHIVE}"
assert RUNNER.is_file(), f"T4ランチャーが見つかりません: {RUNNER}"
assert not OUT.exists(), f"同名の実験フォルダーが既にあります。元データを上書きしないでください: {OUT}"

cmd = [
    sys.executable, "-u", str(RUNNER),
    "--study", str(STUDY),
    "--v47-zip", str(ARCHIVE),
    "--output", str(OUT),
    "--minimum-updates", "10000",
    "--maximum-updates", "20000",
    "--execute",
]
print("旧全教材の絶対教師トークン数を維持できるか検査し、通過した場合だけT4学習を開始します。", flush=True)

# Child-process errors were hidden by a generic CalledProcessError.
# Stream stdout+stderr into a permanent Drive diagnostic and display the
# complete last traceback on failures. The same training command is retained.
from collections import deque
from datetime import datetime, timezone
import re
import uuid

DIAG = STUDY / "_v48_t4_diagnostics"
DIAG.mkdir(parents=True, exist_ok=True)
LOG = DIAG / (
    "v48_t4_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") +
    "_" + uuid.uuid4().hex[:8] + ".log"
)
last_lines = deque(maxlen=80)
show = re.compile(
    r"Traceback|Error|Exception|FAILED|V48_|PREFLIGHT|GECPU_|"
    r"RuntimeError|FileNotFoundError|ValueError|^\\s*File |"
    r"checkpoint|SHA256|new dataset|new curriculum|T4|^\\s*Step ",
    re.IGNORECASE,
)
seen = 0
with LOG.open("x", encoding="utf-8") as log:
    with subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    ) as child:
        for line in child.stdout:
            log.write(line)
            last_lines.append(line.rstrip())
            seen += 1
            if seen <= 25 or show.search(line) or seen % 250 == 0:
                print(line, end="", flush=True)
            if seen % 20 == 0:
                log.flush()
        code = child.wait()
    log.flush()

print("\n全実行ログ:", LOG, flush=True)
if code != 0:
    print("\n=== V48_CHILD_ERROR: subprocess exit", code, "===", flush=True)
    print("\n".join(last_lines), flush=True)
    raise RuntimeError(
        f"学習ランチャーが終了コード{code}で停止しました。"
        f"直前のV48_CHILD_ERRORとログ {LOG} に本当の原因があります。"
        " GPU学習完走は確認されていません。"
    )
print("学習処理終了。出力先:", OUT)
